In [1]:
from pathlib import Path
from collections import Counter

ROOTS = [Path(r"C:\Users\ebteda\Desktop\CSD"),
         Path(r"C:\Users\ebteda\Desktop\CSD_reannotation-main")]

for root in ROOTS:
    print("=" * 70)
    print("ROOT:", root, "| exists:", root.exists())
    if not root.exists():
        continue
    all_files = [p for p in root.rglob("*") if p.is_file()]
    print("total files:", len(all_files))
    print("by type:", dict(Counter(p.suffix.lower() for p in all_files)))

    # every folder that contains files: how many, what types, 3 example names
    folders = sorted({p.parent for p in all_files})
    for f in folders:
        files = [p for p in all_files if p.parent == f]
        types = dict(Counter(p.suffix.lower() for p in files))
        print(f"\n  {f.relative_to(root) if f != root else '.'}  ->  {len(files)} files {types}")
        for p in sorted(files)[:3]:
            print("     e.g.", p.name)

ROOT: C:\Users\ebteda\Desktop\CSD | exists: True
total files: 1002
by type: {'.json': 2, '.csv': 200, '.txt': 400, '.mid': 200, '.wav': 200}

  CSD\english  ->  1 files {'.json': 1}
     e.g. metadata.json

  CSD\english\csv  ->  100 files {'.csv': 100}
     e.g. en001a.csv
     e.g. en001b.csv
     e.g. en002a.csv

  CSD\english\lyric  ->  100 files {'.txt': 100}
     e.g. en001a.txt
     e.g. en001b.txt
     e.g. en002a.txt

  CSD\english\mid  ->  100 files {'.mid': 100}
     e.g. en001a.mid
     e.g. en001b.mid
     e.g. en002a.mid

  CSD\english\txt  ->  100 files {'.txt': 100}
     e.g. en001a.txt
     e.g. en001b.txt
     e.g. en002a.txt

  CSD\english\wav  ->  100 files {'.wav': 100}
     e.g. en001a.wav
     e.g. en001b.wav
     e.g. en002a.wav

  CSD\korean  ->  1 files {'.json': 1}
     e.g. metadata.json

  CSD\korean\csv  ->  100 files {'.csv': 100}
     e.g. kr001a.csv
     e.g. kr001b.csv
     e.g. kr002a.csv

  CSD\korean\lyric  ->  100 files {'.txt': 100}
     e.g. kr00

In [6]:
import json, wave

CSD = Path(r"C:\Users\ebteda\Desktop\CSD")

def show(path, n=8):
    print(f"\n--- {path.relative_to(path.parents[2])} ---")
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print(f"({len(lines)} lines)")
    for ln in lines[:n]:
        print("  ", ln)

# find the re-annotation folder automatically (it is nested)
RE = next(Path(r"C:\Users\ebteda\Desktop\CSD_reannotation-main").rglob("note")).parent
print("re-annotation folder:", RE)

# 2) re-annotation of the same Korean song
show(RE / "note" / "kr001a.csv")

# 3) lyric/phoneme txt (just to know what it is)
show(CSD / "korean" / "txt" / "kr001a.txt", n=3)

# 4) audio format
for p in [CSD / "english" / "wav" / "en001a.wav", CSD / "korean" / "wav" / "kr001a.wav"]:
    with wave.open(str(p)) as w:
        print(f"\n{p.name}: sr {w.getframerate()} | channels {w.getnchannels()} | "
              f"bits {8*w.getsampwidth()} | {w.getnframes()/w.getframerate():.1f} s")

# 5) metadata (first entry)
meta = json.loads((CSD / "english" / "metadata.json").read_text(encoding="utf-8"))
print("\nmetadata type:", type(meta).__name__, "| size:", len(meta))
first = next(iter(meta.items())) if isinstance(meta, dict) else meta[0]
print("first entry:", first)

re-annotation folder: C:\Users\ebteda\Desktop\CSD_reannotation-main\CSD_reannotation-main\CSD_reannotation

--- CSD_reannotation\note\kr001a.csv ---
(106 lines)
   0.923083900,59,0.194376417
   1.128344671,59,0.224965986
   1.393197279,59,0.286621315
   1.846145125,66,0.409092971
   2.304580499,66,0.464648526
   2.769229025,68,0.440929705
   3.213061224,68,0.101587302
   3.439455782,68,0.148027211

--- korean\txt\kr001a.txt ---
(15 lines)
   e i b_i ss_i d_i i e p_eu j_i
   e i ch_i a i j_e i k_e i e_l e_m e_n
   o p_i k_y_u a_l e s_eu t_i y_u

en001a.wav: sr 44100 | channels 2 | bits 16 | 64.8 s

kr001a.wav: sr 44100 | channels 2 | bits 16 | 55.4 s

metadata type: dict | size: 100
first entry: ('en001a', {'songname': 'Alphabet', 'key_diff': 0, 'tempo': 100, 'time_signature': '4/4'})


In [7]:
import numpy as np, pandas as pd

def read_orig(p):   return pd.read_csv(p)                     # start,end,pitch,syllable
def read_re(p):
    a = np.loadtxt(p, delimiter=",", ndmin=2)                  # onset,pitch,duration
    return pd.DataFrame({"start": a[:, 0], "end": a[:, 0] + a[:, 2], "pitch": a[:, 1]})

meta = {}
for lang in ["english", "korean"]:
    meta.update(json.loads((CSD / lang / "metadata.json").read_text(encoding="utf-8")))

# ---- 1) grid check on ALL 200 original files ----
# share of onsets that sit within 5 ms of a 16th-note grid (from the song's tempo)
rows = []
for lang in ["english", "korean"]:
    for p in sorted((CSD / lang / "csv").glob("*.csv")):
        df = read_orig(p)
        tempo = meta[p.stem]["tempo"]
        step = 60 / tempo / 4                                  # 16th note in seconds
        on = df["start"].values
        rel = (on - on[0]) / step                              # grid anchored at first note
        on_grid = np.mean(np.abs(rel - np.round(rel)) * step < 0.005)
        rows.append({"file": p.stem, "lang": lang, "notes": len(df), "on_grid": on_grid})
grid = pd.DataFrame(rows)
print("=== ORIGINAL labels: share of onsets sitting exactly on the tempo grid ===")
print(grid.groupby("lang")["on_grid"].describe()[["mean", "min", "50%", "max"]].round(2))

# ---- 2) original vs re-annotation, the 50 Korean 'a' files ----
def match_stats(a, b, tol):
    """F1 between two onset lists (seconds), greedy one-to-one within tol."""
    used, tp = set(), 0
    for t in a:
        j = next((k for k, u in enumerate(b) if abs(u - t) <= tol and k not in used), None)
        if j is not None: tp += 1; used.add(j)
    p, r = tp / len(b), tp / len(a)
    return 2*p*r/(p+r) if p+r else 0.0

cmp_rows, diffs = [], []
for p in sorted((RE / "note").glob("*.csv")):
    o = read_orig(CSD / "korean" / "csv" / p.name)["start"].values
    r = read_re(p)["start"].values
    cmp_rows.append({"file": p.stem, "notes_orig": len(o), "notes_re": len(r),
                     "F1_20": match_stats(o, r, 0.02), "F1_50": match_stats(o, r, 0.05)})
    for t in r:                                               # re - nearest original, within 200 ms
        d = t - o[np.argmin(np.abs(o - t))]
        if abs(d) <= 0.2: diffs.append(d * 1000)
cmp = pd.DataFrame(cmp_rows); d = np.array(diffs)
print("\n=== ORIGINAL vs RE-ANNOTATION (50 Korean files) ===")
print(f"same note count in {np.mean(cmp.notes_orig == cmp.notes_re):.0%} of files "
      f"| total notes orig {cmp.notes_orig.sum()} vs re {cmp.notes_re.sum()}")
print(f"onset agreement: F1 ±20ms {cmp.F1_20.mean():.3f} | ±50ms {cmp.F1_50.mean():.3f}")
print(f"re - orig: median {np.median(d):+.0f} ms | typical gap {np.median(np.abs(d)):.0f} ms")

=== ORIGINAL labels: share of onsets sitting exactly on the tempo grid ===
         mean   min  50%  max
lang                         
english  0.42  0.02  0.2  1.0
korean   0.91  0.10  1.0  1.0

=== ORIGINAL vs RE-ANNOTATION (50 Korean files) ===
same note count in 90% of files | total notes orig 7908 vs re 7897
onset agreement: F1 ±20ms 0.502 | ±50ms 0.881
re - orig: median +5 ms | typical gap 21 ms


In [8]:
# ---- English: how grid-like is each file? ----
eng = grid[grid.lang == "english"].copy()
bins = [0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.01]
print("=== English files by share of onsets on the grid ===")
print(pd.cut(eng.on_grid, bins, right=False).value_counts().sort_index().to_string())

GRID_MAX = 0.30                                    # keep files with < 30% of onsets on the grid
eng_keep = eng[eng.on_grid < GRID_MAX]
print(f"\nEnglish kept (< {GRID_MAX:.0%} on grid): {len(eng_keep)} of {len(eng)} files "
      f"| key a: {eng_keep.file.str.endswith('a').sum()} | key b: {eng_keep.file.str.endswith('b').sum()}")

# ---- final clean list ----
clean = []
for f in eng_keep.file:                            # English: original labels
    clean.append({"file": f, "lang": "english", "song": f[:-1],
                  "wav": CSD / "english" / "wav" / f"{f}.wav",
                  "label": CSD / "english" / "csv" / f"{f}.csv", "label_type": "original"})
for p in sorted((RE / "note").glob("*.csv")):      # Korean: re-annotation (key a only)
    clean.append({"file": p.stem, "lang": "korean", "song": p.stem[:-1],
                  "wav": CSD / "korean" / "wav" / f"{p.stem}.wav",
                  "label": p, "label_type": "reannotation"})
clean = pd.DataFrame(clean)

# ---- checks ----
def n_notes(r):
    return len(read_orig(r.label)) if r.label_type == "original" else len(read_re(r.label))
clean["notes"] = clean.apply(n_notes, axis=1)
clean["wav_exists"] = clean.wav.apply(lambda p: p.exists())
print("\n=== CLEAN CSD ===")
print(clean.groupby("lang").agg(files=("file", "size"), songs=("song", "nunique"), notes=("notes", "sum")))
print("total files:", len(clean), "| total notes:", clean.notes.sum(),
      "| all wavs found:", clean.wav_exists.all())
print("songs present in both keys (a+b):", (clean.groupby("song").size() == 2).sum(),
      "-> must stay on the same side of any train/test split")

=== English files by share of onsets on the grid ===
on_grid
[0.0, 0.1)     26
[0.1, 0.2)     25
[0.2, 0.3)      8
[0.3, 0.5)      5
[0.5, 0.8)      7
[0.8, 1.01)    29

English kept (< 30% on grid): 59 of 100 files | key a: 29 | key b: 30

=== CLEAN CSD ===
         files  songs  notes
lang                        
english     59     32   8846
korean      50     50   7897
total files: 109 | total notes: 16743 | all wavs found: True
songs present in both keys (a+b): 27 -> must stay on the same side of any train/test split


In [9]:
import shutil

OUT = Path(r"C:\Users\ebteda\Desktop\csd_upload")
if OUT.exists(): shutil.rmtree(OUT)
(OUT / "audio").mkdir(parents=True); (OUT / "annot").mkdir()

for r in clean.itertuples():
    shutil.copy(r.wav, OUT / "audio" / f"{r.file}.wav")
    df = read_orig(r.label) if r.label_type == "original" else read_re(r.label)
    df[["start", "end", "pitch"]].to_csv(OUT / "annot" / f"{r.file}.csv", index=False)

# small index file: which file, language, song, where the label came from
clean[["file", "lang", "song", "label_type", "notes"]].to_csv(OUT / "index.csv", index=False)

# ---- checks ----
print("audio files:", len(list((OUT / "audio").glob("*.wav"))), "(expect 109)")
print("label files:", len(list((OUT / "annot").glob("*.csv"))), "(expect 109)")
print("example label:\n", (OUT / "annot" / "kr001a.csv").read_text().splitlines()[:3])

shutil.make_archive(str(OUT), "zip", OUT)
print("\nzip ready:", str(OUT) + ".zip")

audio files: 109 (expect 109)
label files: 109 (expect 109)
example label:
 ['start,end,pitch', '0.9230839,1.117460317,59.0', '1.128344671,1.353310657,59.0']

zip ready: C:\Users\ebteda\Desktop\csd_upload.zip
